In [39]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Step 1: Load dataset

In [40]:
df=pd.read_csv("StudentPerformance.csv")

print("Data Loaded successfully")
print("Shape of dataset:",df.shape)


Data Loaded successfully
Shape of dataset: (6607, 20)


In [41]:
df.head(5)

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [42]:
df.columns

Index(['Hours_Studied', 'Attendance', 'Parental_Involvement',
       'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours',
       'Previous_Scores', 'Motivation_Level', 'Internet_Access',
       'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type',
       'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities',
       'Parental_Education_Level', 'Distance_from_Home', 'Gender',
       'Exam_Score'],
      dtype='object')

# Step 2: Split the data into X and y

In [43]:
X=df.drop("Exam_Score",axis=1,errors='ignore')
y=df["Exam_Score"]

print("X shape:" ,X.shape)
print("y shape:" ,y.shape)

X shape: (6607, 19)
y shape: (6607,)


# Step 3: Split the data into train and test

In [44]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

print("Train shape:",X_train.shape)
print("Test shape:",X_test.shape)

Train shape: (5285, 19)
Test shape: (1322, 19)


# step 3: Feature Engineering on train data

In [45]:
from sklearn.preprocessing import LabelEncoder,MinMaxScaler

#fill the missing value
for col in X_train.columns:
    if X_train[col].dtype=="object":
        X_train[col]=X_train[col].fillna(X_train[col].mode()[0])
    else:
        X_train[col]=X_train[col].fillna(X_train[col].median())
        
# split the cols into cat_cols and num_cols
cat_cols=X_train.select_dtypes(include=['object']).columns
num_cols=X_train.select_dtypes(exclude=['object']).columns

# label encoding for the cat_cols
label_encoders={}
for col in cat_cols:
    le=LabelEncoder()
    X_train[col]=le.fit_transform(X_train[col])
    label_encoders[col]=le

num_cols=X_train.columns

#Normalization for numerical features
scalerr=MinMaxScaler()
X_train[num_cols]=scalerr.fit_transform(X_train[num_cols])
print("Feature engineering on train data completed")
    


Feature engineering on train data completed


# Step 4 Model building

In [46]:
from sklearn.linear_model import LinearRegression

model=LinearRegression()

model.fit(X_train,y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


# Step 5 Feature Engineering on test data

In [47]:
# fill missing values (use TRAIN values)
for col in X_test.columns:
    if X_test[col].dtype == "object":
        X_test[col] = X_test[col].fillna(X_train[col].mode()[0])
    else:
        X_test[col] = X_test[col].fillna(X_train[col].median())

# encoding
for col in cat_cols:
    le = label_encoders[col]
    X_test[col] = X_test[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
    X_test[col] = le.transform(X_test[col])

X_test = X_test[X_train.columns]
# scaling
X_test[num_cols] = scalerr.transform(X_test[num_cols])

print("Test data ready")

Test data ready


# Step6: prediction on TestData

In [48]:
y_pred=model.predict(X_test)
print("predictions generated")

predictions generated


# Step:7 Model Evaluation

In [49]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

MSE: 4.4224022406453125
MAE: 1.0251834119912302
R2 Score: 0.6871326801418753


# Step:8 model deployment

In [51]:
import pickle

# save model
pickle.dump(model, open("model.pkl", "wb"))

# save scaler
pickle.dump(scalerr, open("scaler.pkl", "wb"))

# save encoders
pickle.dump(label_encoders, open("encoders.pkl", "wb"))

print("Saved successfully")

Saved successfully
